### Labelled-data curve

Fine-tuned RoBERTa-large on stratified subsets of each training partition, scored
on the full test split. Two reference lines mark what is available without
fine-tuning at all, the frozen probe and the rule-based dictionary.

Reads `results.csv`, writes `data_curve.png`.


In [ ]:
import sys

sys.path.insert(0, "..")
from config import RESULTS_DIR

import matplotlib.pyplot as plt
import polars as pl

# match the thesis body font (Computer Modern)
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["cmr10", "DejaVu Serif"],
        "mathtext.fontset": "cm",
        "axes.unicode_minus": False,
    }
)

INK, GREYT = "#2b3440", "#5b6675"
LINE, REF = "#009E73", "#D55E00"

d = (
    pl.read_csv(RESULTS_DIR / "results.csv")
    .filter(pl.col("corpus") == "twd")
    .group_by("model")
    .agg(pl.col("macro_f1").mean().alias("m"), pl.col("macro_f1").std(ddof=0).alias("s"))
)
m = {r["model"]: (r["m"], r["s"]) for r in d.to_dicts()}

# read whatever subset rows exist, so the figure follows the experiment
subset = sorted(
    (int(k.split("-")[1].split(":")[0]), k) for k in m if k.startswith("subset-")
)
SIZES = [n for n, _ in subset] + [1984]
KEYS = [k for _, k in subset] + ["roberta-large"]
ys = [m[k][0] for k in KEYS]
es = [m[k][1] for k in KEYS]

frozen = m["frozen:roberta-large"][0]
rules = m["rule-based"][0]

fig, ax = plt.subplots(figsize=(6.2, 4.3))

# reference lines: what you get without fine-tuning at all
ax.axhline(frozen, color=REF, lw=1.2, ls="--", zorder=1)
ax.text(132, frozen + 0.008, f"frozen probe, {frozen:.3f}", fontsize=11.5, color=REF)
ax.axhline(rules, color=GREYT, lw=1.2, ls=":", zorder=1)
ax.text(2300, rules + 0.008, f"rule-based dictionary, {rules:.3f}", fontsize=11.5, color=GREYT, ha="right")

ax.errorbar(
    SIZES, ys, yerr=es, color=LINE, lw=1.8, marker="o", ms=6,
    capsize=3, capthick=1.2, elinewidth=1.2, zorder=3,
)
ax.set_xscale("log")
ticks = [125, 250, 500, 1000, 1984]
ax.set_xticks(ticks)
ax.set_xticklabels([f"{n:,}" for n in ticks], fontsize=11.5)
ax.minorticks_off()
ax.set_xlim(110, 2350)
ax.set_ylim(0.40, 0.79)
ax.set_xlabel("labelled training sentences", fontsize=13, color=INK, fontname="cmb10")
ax.set_ylabel("macro-F1 on test (mean of 3 seeds)", fontsize=13, color=INK, fontname="cmb10")
ax.tick_params(axis="y", labelsize=11.5)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig("data_curve.png", dpi=200, bbox_inches="tight", facecolor="white")
fig.savefig("data_curve.pdf", bbox_inches="tight", facecolor="white")
print({n: round(y, 4) for n, y in zip(SIZES, ys)})
